# VOLTA — EDA & Training Walkthrough

This notebook is a guided tour of the VOLTA pipeline. The reusable logic lives in
`../ml/` (so it can run headless on Colab / CI); here we explore the data and the
results interactively.

**Pipeline:** `data.py` → `train.py` → `baselines.py` → `export.py`

> macOS note: PyTorch and XGBoost deadlock if imported in the same process
> (duplicate OpenMP). Keep the XGBoost baseline in its own process.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path('../ml').resolve()))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from data import load_energy
from config import REGIONS, DEFAULT_REGION

df = load_energy()
df.head()

## 1. EDA — load profiles, seasonality, temperature response

In [ ]:
d = df[df.region == DEFAULT_REGION].set_index('datetime')
fig, ax = plt.subplots(2, 1, figsize=(12, 6))
d['load_mw'].iloc[:24*14].plot(ax=ax[0], title=f'{DEFAULT_REGION} — two weeks of hourly load')
ax[1].scatter(d['temperature'], d['load_mw'], s=2, alpha=0.2)
ax[1].set(xlabel='temperature (°C)', ylabel='load (MW)', title='U-shaped temperature response')
plt.tight_layout()

In [ ]:
# Average daily shape by weekday vs weekend
d['hour'] = d.index.hour; d['weekend'] = d.index.dayofweek >= 5
piv = d.groupby(['hour', 'weekend'])['load_mw'].mean().unstack()
piv.plot(figsize=(10, 4), title='Mean load by hour: weekday vs weekend')
plt.ylabel('load (MW)')

## 2. Train + evaluate

Run from a shell (kept out-of-process for the OpenMP reason above):
```bash
cd ../ml
python train.py        # CNN-BiLSTM (pooled, all regions) + seasonal-naive
python baselines.py    # XGBoost
python export.py       # -> web/public/{models,data}
```

## 3. Inspect the exported scorecard

In [ ]:
import json
scorecard = json.load(open('../web/public/data/metrics.json'))
pd.DataFrame(scorecard[DEFAULT_REGION]['models']).T.round(3)